# Fase 3 · M03: Features Temporales y Derivadas

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 3 — Feature Engineering |
| **Módulo** | M03 — Features |

---

## 🎯 Qué hace

Genera features temporales y derivadas a partir del expediente académico: años de gap, indicadores de interrupción, ratios de progreso.

## 📋 Requisitos

- `data/03_features/df_expediente_base.parquet`
- `data/03_features/df_alumno_limpio.parquet`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/03_features/df_expediente_features.parquet` | Dataset con features derivadas |
| `docs/html/fase3/m03_features.html` | Informe HTML |

## 🔄 Flujo

```
df_expediente_base.parquet + df_alumno_limpio.parquet
    ↓ Features temporales (gap, interrupción)
    ↓ Features de progreso (ratio créditos)
    → data/03_features/df_expediente_features.parquet + HTML
```

## ➡️ Siguiente

`f3_m04a_automl_target.ipynb` — construcción del target y dataset para AutoML


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')

# Detectar entorno
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import RUTA_FEATURES, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es, formato_porcentaje_es
from src.utils.graficos import figura_a_base64, COLORES
from src.html import (
    generar_kpis_html,
    generar_seccion_html,
    generar_html_navegacion_completa,
    guardar_html
)
from src.html.render import render_base_html

# Rutas
RUTA_FASE3_HTML = RUTA_HTML / 'fase3'
crear_directorios([RUTA_FEATURES, RUTA_FASE3_HTML])

info_entorno()

✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATOS
# ============================================================================

print('=' * 60)
print('F3-M03: FEATURES TEMPORALES Y DERIVADAS')
print('=' * 60)

df = pd.read_parquet(RUTA_FEATURES / 'df_expediente_base.parquet')
fmt = formato_numero_es

n_exp = len(df)
n_cols_antes = len(df.columns)
cols_antes = set(df.columns)

print(f'📥 Cargado: {fmt(n_exp)} expedientes × {n_cols_antes} columnas')

F3-M03: FEATURES TEMPORALES Y DERIVADAS
📥 Cargado: 33.621 expedientes × 39 columnas


In [3]:
# ============================================================================
# CELDA 3: FEATURES TEMPORALES
# ============================================================================

print('\n' + '=' * 60)
print('FEATURES TEMPORALES')
print('=' * 60)

# 3.1 Duración real (años desde inicio hasta último curso)
df['duracion_real'] = df['curso_ultimo'] - df['curso_inicio'] + 1
print(f'✅ duracion_real: rango {df["duracion_real"].min()}-{df["duracion_real"].max()} años')

# 3.2 Años con gaps (duración - cursos matriculados)
df['anios_gap'] = df['duracion_real'] - df['n_cursos']
df['tiene_gaps'] = df['anios_gap'] > 0
n_con_gaps = df['tiene_gaps'].sum()
print(f'✅ tiene_gaps: {fmt(n_con_gaps)} expedientes ({n_con_gaps/n_exp*100:.1f}%)')

# 3.3 Ratio de avance (cursos matriculados / duración real)
df['ratio_avance'] = np.where(
    df['duracion_real'] > 0,
    df['n_cursos'] / df['duracion_real'],
    1
)
print(f'✅ ratio_avance: media {df["ratio_avance"].mean():.2f}')

# 3.4 Velocidad de créditos (créditos superados / duración real)
df['velocidad_creditos'] = np.where(
    df['duracion_real'] > 0,
    df['cred_superados_total'] / df['duracion_real'],
    df['cred_superados_total']
)
print(f'✅ velocidad_creditos: media {df["velocidad_creditos"].mean():.1f} créd/año')


FEATURES TEMPORALES
✅ duracion_real: rango 1-11 años
✅ tiene_gaps: 1.021 expedientes (3.0%)
✅ ratio_avance: media 0.99
✅ velocidad_creditos: media 44.1 créd/año


In [4]:
# ============================================================================
# CELDA 4: FEATURES DE EVOLUCIÓN
# ============================================================================

print('\n' + '=' * 60)
print('FEATURES DE EVOLUCIÓN')
print('=' * 60)

# 4.1 Mejora de notas (último año - primer año)
df['mejora_notas'] = df['nota_ultimo_anio'] - df['nota_1er_anio']
n_mejoraron = (df['mejora_notas'] > 0).sum()
n_empeoraron = (df['mejora_notas'] < 0).sum()
print(f'✅ mejora_notas: {fmt(n_mejoraron)} mejoraron, {fmt(n_empeoraron)} empeoraron')

# 4.2 Porcentaje de titulación completada (calculado)
df['pct_titulacion'] = (df['cred_superados_total'] / df['cred_titulacion'] * 100).clip(0, 100)
print(f'✅ pct_titulacion: media {df["pct_titulacion"].mean():.1f}%')

# 4.3 Progreso relativo (% titulación / duración esperada)
# Asumimos duración esperada = 4 años para grados
DURACION_ESPERADA = 4
df['progreso_relativo'] = np.where(
    df['duracion_real'] > 0,
    df['pct_titulacion'] / (df['duracion_real'] / DURACION_ESPERADA * 100),
    df['pct_titulacion']
)
df['progreso_relativo'] = df['progreso_relativo'].clip(upper=200)
print(f'✅ progreso_relativo: media {df["progreso_relativo"].mean():.1f}%')

# 4.4 Créditos faltantes
df['cred_faltantes'] = df['cred_titulacion'] - df['cred_superados_total']
df['cred_faltantes'] = df['cred_faltantes'].clip(lower=0)
print(f'✅ cred_faltantes: media {df["cred_faltantes"].mean():.1f} créditos')



FEATURES DE EVOLUCIÓN
✅ mejora_notas: 15.253 mejoraron, 5.924 empeoraron
✅ pct_titulacion: media 60.0%
✅ progreso_relativo: media 0.7%
✅ cred_faltantes: media 97.7 créditos


In [5]:
# ============================================================================
# CELDA 5: FEATURES DE ESTABILIDAD (requiere datos longitudinales)
# ============================================================================

print('\n' + '=' * 60)
print('FEATURES DE ESTABILIDAD')
print('=' * 60)

# Para calcular estabilidad, necesitamos los datos originales
# Cargamos df_alumno_limpio para calcular std de notas por expediente

df_alumno = pd.read_parquet(RUTA_FEATURES / 'df_alumno_limpio.parquet')

# 5.1 Estabilidad académica (std de notas por expediente)
if 'media_curso' in df_alumno.columns:
    estabilidad = df_alumno.groupby(['per_id_ficticio', 'exp_tit_id'])['media_curso'].std().reset_index()
    estabilidad.columns = ['per_id_ficticio', 'exp_tit_id', 'estabilidad_academica']
    df = df.merge(estabilidad, on=['per_id_ficticio', 'exp_tit_id'], how='left')
    df['estabilidad_academica'] = df['estabilidad_academica'].fillna(0)
    print(f'✅ estabilidad_academica: media {df["estabilidad_academica"].mean():.2f}')
else:
    df['estabilidad_academica'] = 0
    print(f'⚠️ estabilidad_academica: no hay datos de media_curso')

# 5.2 Años sin beca
if 'n_anios_beca' in df.columns and 'n_cursos' in df.columns:
    df['anios_sin_beca'] = df['n_cursos'] - df['n_anios_beca']
    df['anios_sin_beca'] = df['anios_sin_beca'].clip(lower=0)
    print(f'✅ anios_sin_beca: media {df["anios_sin_beca"].mean():.1f}')
else:
    df['anios_sin_beca'] = df['n_cursos']

# Liberar memoria
del df_alumno


FEATURES DE ESTABILIDAD


✅ estabilidad_academica: media 0.42
✅ anios_sin_beca: media 1.5


In [6]:
# ============================================================================
# CELDA 6: RESUMEN DE FEATURES NUEVAS
# ============================================================================

print('\n' + '=' * 60)
print('RESUMEN DE FEATURES')
print('=' * 60)

n_cols_despues = len(df.columns)
cols_nuevas = sorted(set(df.columns) - cols_antes)

print(f'📊 Columnas antes: {n_cols_antes}')
print(f'📊 Columnas después: {n_cols_despues}')
print(f'📊 Features nuevas: {len(cols_nuevas)}')
print(f'\n📋 Lista de features nuevas:')
for col in cols_nuevas:
    print(f'   - {col}')

# Resumen estadístico de features nuevas
features_info = []
for col in cols_nuevas:
    if df[col].dtype in ['float64', 'int64', 'float32', 'int32']:
        features_info.append({
            'nombre': col,
            'tipo': 'numérica',
            'min': df[col].min(),
            'max': df[col].max(),
            'mean': df[col].mean(),
            'nulos': df[col].isna().sum()
        })
    else:
        features_info.append({
            'nombre': col,
            'tipo': 'categórica/bool',
            'valores': df[col].nunique(),
            'nulos': df[col].isna().sum()
        })


RESUMEN DE FEATURES
📊 Columnas antes: 39
📊 Columnas después: 50
📊 Features nuevas: 11

📋 Lista de features nuevas:
   - anios_gap
   - anios_sin_beca
   - cred_faltantes
   - duracion_real
   - estabilidad_academica
   - mejora_notas
   - pct_titulacion
   - progreso_relativo
   - ratio_avance
   - tiene_gaps
   - velocidad_creditos


In [7]:
# ============================================================================
# CELDA 7: GRÁFICOS DE FEATURES
# ============================================================================

print('\n' + '=' * 60)
print('GENERANDO GRÁFICOS')
print('=' * 60)

graficos_base64 = {}

# G1: Distribución de duración real
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df['duracion_real'], bins=15, color=COLORES['primary'], edgecolor='white', alpha=0.8)
ax.axvline(DURACION_ESPERADA, color=COLORES['success'], linestyle='--', linewidth=2, label=f'Esperada: {DURACION_ESPERADA} años')
ax.set_xlabel('Duración (años)')
ax.set_ylabel('Frecuencia')
ax.set_title('Duración Real del Expediente', fontweight='bold')
ax.legend()
plt.tight_layout()
graficos_base64['duracion'] = figura_a_base64(fig)
plt.close()

# G2: Distribución de mejora de notas
fig, ax = plt.subplots(figsize=(6, 4))
mejora_valida = df['mejora_notas'].dropna()
colores_mejora = np.where(mejora_valida >= 0, COLORES['success'], COLORES['danger'])
ax.hist(mejora_valida, bins=30, color=COLORES['warning'], edgecolor='white', alpha=0.8)
ax.axvline(0, color='black', linestyle='-', linewidth=1)
ax.axvline(mejora_valida.mean(), color=COLORES['danger'], linestyle='--', linewidth=2, label=f'Media: {mejora_valida.mean():.2f}')
ax.set_xlabel('Mejora de notas (último - primero)')
ax.set_ylabel('Frecuencia')
ax.set_title('Evolución de Notas', fontweight='bold')
ax.legend()
plt.tight_layout()
graficos_base64['mejora'] = figura_a_base64(fig)
plt.close()

# G3: Distribución de estabilidad académica
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df['estabilidad_academica'], bins=25, color=COLORES['primary'], edgecolor='white', alpha=0.8)
ax.set_xlabel('Desviación típica de notas')
ax.set_ylabel('Frecuencia')
ax.set_title('Estabilidad Académica', fontweight='bold')
plt.tight_layout()
graficos_base64['estabilidad'] = figura_a_base64(fig)
plt.close()

# G4: Barras tiene_gaps
fig, ax = plt.subplots(figsize=(5, 4))
valores_gaps = df['tiene_gaps'].value_counts().sort_index()
labels = ['Sin gaps', 'Con gaps']
colores_bars = [COLORES['success'], COLORES['warning']]
bars = ax.bar(labels, valores_gaps.values, color=colores_bars)
for bar, val in zip(bars, valores_gaps.values):
    pct = val / n_exp * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + n_exp*0.01, 
            f'{fmt(val)}\n({pct:.1f}%)', ha='center', fontsize=9)
ax.set_ylabel('Expedientes')
ax.set_title('Interrupciones en Trayectoria', fontweight='bold')
plt.tight_layout()
graficos_base64['gaps'] = figura_a_base64(fig)
plt.close()

# G5: Distribución de velocidad de créditos
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df['velocidad_creditos'], bins=30, color=COLORES['success'], edgecolor='white', alpha=0.8)
ax.axvline(60, color=COLORES['primary'], linestyle='--', linewidth=2, label='Esperado: 60 créd/año')
ax.set_xlabel('Créditos por año')
ax.set_ylabel('Frecuencia')
ax.set_title('Velocidad de Avance', fontweight='bold')
ax.legend()
plt.tight_layout()
graficos_base64['velocidad'] = figura_a_base64(fig)
plt.close()

# G6: Boxplot comparativo de features temporales
fig, ax = plt.subplots(figsize=(8, 4))
datos_box = [
    df['duracion_real'],
    df['n_cursos'],
    df['anios_gap']
]
bp = ax.boxplot(datos_box, labels=['Duración real', 'Cursos matr.', 'Años gap'], patch_artist=True)
colores_box = [COLORES['primary'], COLORES['success'], COLORES['warning']]
for patch, color in zip(bp['boxes'], colores_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel('Años')
ax.set_title('Comparativa Temporal', fontweight='bold')
plt.tight_layout()
graficos_base64['temporal_box'] = figura_a_base64(fig)
plt.close()

print(f'✅ {len(graficos_base64)} gráficos generados')


GENERANDO GRÁFICOS


✅ 6 gráficos generados


In [8]:
# ============================================================================
# CELDA 7.5: AGREGAR VARIABLES CATEGÓRICAS (forma_de_pago)
# ============================================================================

print('\n' + '=' * 70)
print('AGREGANDO VARIABLES CATEGÓRICAS')
print('=' * 70)

# Leer forma_de_pago de df_expediente_base original
df_base_original = pd.read_parquet(RUTA_FEATURES / 'df_expediente_base.parquet')

if 'forma_de_pago' in df_base_original.columns:
    print('\n1️⃣  forma_de_pago')
    df['forma_de_pago'] = df_base_original['forma_de_pago']
    print(f'   ✅ Valores: {df["forma_de_pago"].unique()}')
    print(f'   ✅ Agregada a dataset')
else:
    print('\n❌ forma_de_pago no encontrada en entrada')

# Verificación
n_cols_despues = len(df.columns)
n_cols_nuevas = n_cols_despues - n_cols_antes
print(f'\n✅ Columnas después: {n_cols_despues} (+{n_cols_nuevas} nuevas)')



AGREGANDO VARIABLES CATEGÓRICAS

❌ forma_de_pago no encontrada en entrada

✅ Columnas después: 50 (+11 nuevas)


In [9]:
# ============================================================================
# CELDA 8: GUARDAR DATASET
# ============================================================================

print('\n' + '=' * 60)
print('GUARDANDO DATASET')
print('=' * 60)

ruta_salida = RUTA_FEATURES / 'df_expediente_features.parquet'
df.to_parquet(ruta_salida, index=False)
tamanio_mb = ruta_salida.stat().st_size / 1024 / 1024
print(f'💾 Guardado: {ruta_salida.name} ({tamanio_mb:.1f} MB)')

# Guardar también CSV
ruta_csv = RUTA_FEATURES / 'df_expediente_features.csv'
df.to_csv(ruta_csv, sep=';', index=False, encoding='utf-8-sig')
print(f'✓ CSV guardado: {ruta_csv.name}')


GUARDANDO DATASET
💾 Guardado: df_expediente_features.parquet (1.6 MB)


✓ CSV guardado: df_expediente_features.csv


In [10]:
# ============================================================================
# CELDA 9: GENERAR HTML
# ============================================================================

print('\n' + '=' * 60)
print('GENERANDO HTML')
print('=' * 60)

nav_fases_html, nav_modulos_html = generar_html_navegacion_completa(
    fase_activa='fase3',
    modulo_activo='m03'
)

# KPIs
KPIS = [
    {'valor': fmt(n_exp), 'titulo': 'Expedientes'},
    {'valor': str(n_cols_despues), 'titulo': 'Columnas'},
    {'valor': f'+{len(cols_nuevas)}', 'titulo': 'Features nuevas'},
    {'valor': f"{df['duracion_real'].mean():.1f}", 'titulo': 'Duración media'},
]
kpis_html = generar_kpis_html(KPIS)

# S1: Features temporales
features_temp = [
    ('duracion_real', 'curso_ultimo - curso_inicio + 1', 'Años totales del expediente'),
    ('anios_gap', 'duracion_real - n_cursos', 'Años sin matrícula'),
    ('tiene_gaps', 'anios_gap > 0', 'Tuvo interrupciones'),
    ('ratio_avance', 'n_cursos / duracion_real', 'Continuidad de matrícula'),
    ('velocidad_creditos', 'cred_superados / duracion_real', 'Créditos por año'),
]
filas_temp = ''.join([f'<tr><td><code>{n}</code></td><td>{f}</td><td>{d}</td></tr>' for n, f, d in features_temp])
s1 = generar_seccion_html('Features Temporales', f'''
<table style="width:100%;border-collapse:collapse;">
<tr style="background:#3182ce;color:white;"><th style="padding:10px;">Feature</th><th>Fórmula</th><th>Descripción</th></tr>
{filas_temp}
</table>
''', '⏱️')

# S2: Features de evolución
features_evol = [
    ('mejora_notas', 'nota_ultimo - nota_1er', 'Cambio en rendimiento'),
    ('progreso_relativo', 'pct_titulacion / (duracion/4 × 100)', 'Avance vs esperado'),
    ('cred_faltantes', 'cred_titulacion - cred_superados', 'Créditos para terminar'),
]
filas_evol = ''.join([f'<tr><td><code>{n}</code></td><td>{f}</td><td>{d}</td></tr>' for n, f, d in features_evol])
s2 = generar_seccion_html('Features de Evolución', f'''
<table style="width:100%;border-collapse:collapse;">
<tr style="background:#38a169;color:white;"><th style="padding:10px;">Feature</th><th>Fórmula</th><th>Descripción</th></tr>
{filas_evol}
</table>
''', '📈')

# S3: Features de estabilidad
features_estab = [
    ('estabilidad_academica', 'std(media_curso)', 'Variabilidad de notas'),
    ('anios_sin_beca', 'n_cursos - n_anios_beca', 'Años sin ayuda económica'),
]
filas_estab = ''.join([f'<tr><td><code>{n}</code></td><td>{f}</td><td>{d}</td></tr>' for n, f, d in features_estab])
s3 = generar_seccion_html('Features de Estabilidad', f'''
<table style="width:100%;border-collapse:collapse;">
<tr style="background:#ed8936;color:white;"><th style="padding:10px;">Feature</th><th>Fórmula</th><th>Descripción</th></tr>
{filas_estab}
</table>
''', '📊')

# S4: Gráficos principales
s4 = generar_seccion_html('Distribuciones de Features', f'''
<div style="display:grid;grid-template-columns:repeat(3,1fr);gap:20px;">
    <div style="text-align:center;"><img src="data:image/png;base64,{graficos_base64['duracion']}" style="max-width:100%;border-radius:8px;"/></div>
    <div style="text-align:center;"><img src="data:image/png;base64,{graficos_base64['mejora']}" style="max-width:100%;border-radius:8px;"/></div>
    <div style="text-align:center;"><img src="data:image/png;base64,{graficos_base64['estabilidad']}" style="max-width:100%;border-radius:8px;"/></div>
</div>
''', '📉')

# S5: Gráficos secundarios
s5 = generar_seccion_html('Análisis de Trayectoria', f'''
<div style="display:grid;grid-template-columns:repeat(3,1fr);gap:20px;">
    <div style="text-align:center;"><img src="data:image/png;base64,{graficos_base64['gaps']}" style="max-width:100%;border-radius:8px;"/></div>
    <div style="text-align:center;"><img src="data:image/png;base64,{graficos_base64['velocidad']}" style="max-width:100%;border-radius:8px;"/></div>
    <div style="text-align:center;"><img src="data:image/png;base64,{graficos_base64['temporal_box']}" style="max-width:100%;border-radius:8px;"/></div>
</div>
''', '🔍')

# S6: Resumen estadístico
stats_html = '<div style="display:grid;grid-template-columns:repeat(4,1fr);gap:15px;">'
stats_data = [
    (f"{df['duracion_real'].mean():.1f}", 'Duración media', COLORES['primary']),
    (f"{df['tiene_gaps'].mean()*100:.1f}%", 'Con gaps', COLORES['warning']),
    (f"{df['mejora_notas'].mean():.2f}", 'Mejora media', COLORES['success']),
    (f"{df['velocidad_creditos'].mean():.0f}", 'Créd/año', COLORES['primary']),
]
for valor, titulo, color in stats_data:
    stats_html += f'''
    <div style="background:{color}22;padding:15px;border-radius:8px;text-align:center;border-left:4px solid {color};">
        <div style="font-size:24px;font-weight:bold;color:{color};">{valor}</div>
        <div style="font-size:12px;color:#4a5568;">{titulo}</div>
    </div>'''
stats_html += '</div>'
s6 = generar_seccion_html('Estadísticas Clave', stats_html, '📋')

# HTML completo
contenido_html = kpis_html + s1 + s2 + s3 + s4 + s5 + s6

html_completo = render_base_html(
    titulo='M03: Features Temporales y Derivadas',
    icono='🧪',
    subtitulo='Fase 3: Feature Engineering | TFM Abandono Universitario',
    nav_fases=nav_fases_html,
    nav_modulos=nav_modulos_html,
    contenido=contenido_html,
    notebook_nombre='f3_m03_features.ipynb',
    notebook_carpeta='fase3_feature_engineering'
)

ruta_html = RUTA_FASE3_HTML / 'm03_features.html'
guardar_html(html_completo, ruta_html)
print(f'🌐 HTML: {ruta_html}')


GENERANDO HTML
✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\m03_features.html
🌐 HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\m03_features.html


In [11]:
# ============================================================================
# CELDA 10: RESUMEN FINAL
# ============================================================================

print('\n' + '=' * 60)
print('✅ F3-M03 COMPLETADO')
print('=' * 60)
print(f'📥 Entrada: {fmt(n_exp)} expedientes × {n_cols_antes} columnas')
print(f'📤 Salida: {fmt(n_exp)} expedientes × {n_cols_despues} columnas')
print(f'📊 Features nuevas: {len(cols_nuevas)}')
print(f'💾 {ruta_salida}')
print(f'🌐 {ruta_html}')
print(f'\n📌 Siguiente: f3_m04_encoding.ipynb')


✅ F3-M03 COMPLETADO
📥 Entrada: 33.621 expedientes × 39 columnas
📤 Salida: 33.621 expedientes × 50 columnas
📊 Features nuevas: 11
💾 C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features\df_expediente_features.parquet
🌐 C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\m03_features.html

📌 Siguiente: f3_m04_encoding.ipynb
